In [ ]:
import keras
from keras import layers
import pickle
from pathlib import Path
import numpy as np

In [ ]:
with Path("processed/token_to_id.pkl").open("rb") as f:
    token_to_id = pickle.load(f)
with Path("processed/id_to_token.pkl").open("rb") as f:
    id_to_token = pickle.load(f)

In [ ]:
with Path("processed/groups.csv").open("r") as f:
    first_line = f.readline()
CONTEXT_SIZE = first_line.count(",") // 2

In [ ]:
VOCAB_SIZE = len(id_to_token)
EMBEDDING_SIZE = 100
RANDOM_WORD_COUNT = 10

In [ ]:
root_input = keras.Input((1,), name="root_input")
context_input = keras.Input(
    (CONTEXT_SIZE * 2 + RANDOM_WORD_COUNT,), name="context_input"
)

target_embedding = layers.Embedding(VOCAB_SIZE, EMBEDDING_SIZE, name="target_embedding")
context_embedding = layers.Embedding(
    VOCAB_SIZE, EMBEDDING_SIZE, name="context_embedding"
)

target_vec = target_embedding(root_input)  # (batch, 1, 300)
context_vec = context_embedding(context_input)  # (batch, 4, 300)

dots = layers.Dot(axes=(2, 2))([target_vec, context_vec])
logits = layers.Flatten()(dots)

output = layers.Activation("sigmoid")(logits)

model = keras.Model([root_input, context_input], outputs=output)
model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=["accuracy"],
)
model.summary()

In [ ]:
model.load_weights("last-model.keras")

In [ ]:
# For SGNS-style training, the target and context matrices are separate spaces.
# Use the target matrix as the exported word representation instead of averaging them.
embeddings = target_embedding.get_weights()[0]

In [ ]:
def normalize(v):
    return v / np.linalg.norm(v)

In [ ]:
def norm_embedding(token: str):
    return normalize(embeddings[token_to_id[token]])

In [ ]:
def get_closest_words(query_embedding, top_k=10, exclude_tokens=()):
    # Cosine similarity is largest for the nearest vectors.
    weights_norm = embeddings / np.maximum(
        np.linalg.norm(embeddings, axis=1, keepdims=True), 1e-12
    )
    query_norm = query_embedding / max(np.linalg.norm(query_embedding), 1e-12)
    similarities = weights_norm @ query_norm

    if isinstance(exclude_tokens, str):
        exclude_tokens = (exclude_tokens,)
    excluded_ids = {token_to_id[token] for token in exclude_tokens}
    for token_id in excluded_ids:
        similarities[token_id] = -np.inf

    top_k = min(top_k, len(similarities) - len(excluded_ids))
    top_ids = np.argpartition(similarities, -top_k)[-top_k:]
    top_ids = top_ids[np.argsort(similarities[top_ids])[::-1]]
    return [(id_to_token[int(i)], float(similarities[i])) for i in top_ids]

In [ ]:
def analogy(a, b, c, top_k=10):
    """Find d for a:b :: c:d using normalized 3CosAdd."""
    query = norm_embedding(b) - norm_embedding(a) + norm_embedding(c)
    return get_closest_words(query, top_k=top_k, exclude_tokens=(a, b, c))

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

words = []
word_embeddings = np.array([norm_embedding(word) for word in words])

pca = PCA(n_components=2)
pca_result = pca.fit_transform(word_embeddings)

sns.scatterplot(x=pca_result[:, 0], y=pca_result[:, 1])
for word, (x, y) in zip(words, pca_result):
    plt.annotate(word, (x, y), xytext=(5, 5), textcoords="offset points")